[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/davis-mironga/kitui-washlab-analysis/blob/main/notebooks/07_Report_Figures.ipynb)


# Notebook 07 — Report Figures
**Project:** WASHLAB Climate-Smart WASH Pilot — Kitui County  
**Analyst:** Davis Mironga  
**Purpose:** Produce all publication-ready figures and standalone deliverable maps for the Phase 1 report.

**Requires:** Outputs from Notebooks 01 to 03 in `Kitui_WASHLAB/outputs/` and GEE Assets.

---
## Phase 1 deliverables produced here

| Figure | Deliverable | Source |
|--------|-------------|--------|
| Fig 1 | Seasonal water availability map | GEE Asset: kitui_jrc_water_seasonality |
| Fig 2 | Vegetation stress and land condition map | GEE Asset: kitui_ndvi_mean + baseline |
| Fig 3 | WASI ward choropleth | Notebook 02 output |
| Fig 4 | WASI component breakdown chart | Notebook 02 output |
| Fig 5 | Hotspot analysis map | Notebook 03 output |

## Phase 2 figures (added once borehole data received)
- Fig 6: Borehole network overview
- Fig 7: Coverage gap map
- Fig 8: Priority site rankings

## Outputs
All figures saved to `Kitui_WASHLAB/outputs/maps/` at 300 DPI.
Report tables saved to `Kitui_WASHLAB/outputs/report/`.


### 1. Setup

Installs required libraries, mounts Google Drive, authenticates GEE, and sets the consistent style parameters used across all figures.
Run this cell first before anything else.


In [ ]:
# ── Setup ─────────────────────────────────────────────────────────────────────
!pip install geopandas matplotlib rasterio earthengine-api geemap requests -q

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.io import MemoryFile
from rasterio.warp import reproject
from rasterio.enums import Resampling
from rasterio.features import geometry_mask
from rasterio.transform import from_bounds
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
from shapely.geometry import mapping
import requests
import warnings
warnings.filterwarnings('ignore')

import ee
import geemap
from google.colab import drive

drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/Kitui_WASHLAB/'
OUT   = DRIVE + 'outputs/'
MAPS  = OUT + 'maps/'
REPT  = OUT + 'report/'

import os
os.makedirs(MAPS, exist_ok=True)
os.makedirs(REPT, exist_ok=True)

# GEE authentication
GEE_PROJECT  = 'kitui-washlab-analysis'
ASSET_FOLDER = f'projects/{GEE_PROJECT}/assets/kitui'
ee.Authenticate()
ee.Initialize(project=GEE_PROJECT)

WGS84  = 'EPSG:4326'
BOUNDS = {'west': 37.5, 'east': 39.2, 'south': -3.1, 'north': 0.0}

# Consistent figure style
plt.rcParams.update({
    'font.family':   'DejaVu Sans',
    'font.size':     10,
    'axes.titlesize': 11,
    'axes.labelsize': 9,
    'savefig.dpi':   300,
    'savefig.bbox':  'tight',
})

def add_north_arrow(ax, x=0.95, y=0.10):
    ax.annotate('N', xy=(x, y), xytext=(x, y - 0.05),
                xycoords='axes fraction', fontsize=12,
                ha='center', fontweight='bold',
                arrowprops=dict(arrowstyle='->', color='black', lw=1.5))

def add_caption(ax, text):
    ax.text(0.5, -0.06, text, transform=ax.transAxes,
            ha='center', va='top', fontsize=7.5,
            style='italic', color='#555555')

print('Setup complete')
print(f'Maps will be saved to: {MAPS}')


### 2. Load Analysis Outputs

Loads ward boundaries, WASI results, and hotspot results from Drive.
Also defines a helper function to download rasters from GEE Assets for the seasonal water and vegetation figures.
Phase 2 outputs (coverage gap, rankings) are loaded if available and skipped otherwise.


In [ ]:
# ── Load analysis outputs ──────────────────────────────────────────────────────

# Ward boundaries
wards = gpd.read_file(DRIVE + 'boundaries/kitui_wards.shp').to_crs(WGS84)
if 'Ward' not in wards.columns:
    for col in ['NAME_3', 'WARD', 'ward', 'NAME']:
        if col in wards.columns:
            wards = wards.rename(columns={col: 'Ward'})
            break

# WASI outputs (Notebook 02)
wasi_table = pd.read_csv(OUT + 'kitui_wasi_ward_table.csv')
wasi_wards = gpd.read_file(OUT + 'kitui_wasi_ward.geojson')

# Hotspot outputs (Notebook 03)
hotspot_wards = gpd.read_file(OUT + 'kitui_hotspot_ward.geojson')

# Phase 2 outputs — load if available, skip if not
has_coverage = os.path.exists(OUT + 'kitui_coverage_ward_table.csv')
has_rankings = os.path.exists(OUT + 'kitui_priority_wards.csv')
has_boreholes = os.path.exists(OUT + 'kitui_boreholes.geojson')

if has_coverage:
    coverage_table = pd.read_csv(OUT + 'kitui_coverage_ward_table.csv')
    coverage_wards = gpd.read_file(OUT + 'kitui_coverage_ward.geojson')
if has_boreholes:
    boreholes = gpd.read_file(OUT + 'kitui_boreholes.geojson')

# Helper: download GEE asset as numpy array
PIXEL_DEG = 500 / 111320
GRID_W = int((BOUNDS['east'] - BOUNDS['west']) / PIXEL_DEG)
GRID_H = int((BOUNDS['north'] - BOUNDS['south']) / PIXEL_DEG)
GRID_TRANS = from_bounds(
    BOUNDS['west'], BOUNDS['south'], BOUNDS['east'], BOUNDS['north'],
    GRID_W, GRID_H
)
GRID_SHAPE = (GRID_H, GRID_W)

# County mask for clean visualisation
county_geom = wards.dissolve().geometry.iloc[0]
county_mask = geometry_mask(
    [mapping(county_geom)],
    transform=GRID_TRANS, invert=True, out_shape=GRID_SHAPE
)

def load_asset(asset_name, download_scale):
    asset_id = f'{ASSET_FOLDER}/{asset_name}'
    image = ee.Image(asset_id)
    url = image.getDownloadURL({
        'scale': download_scale, 'crs': WGS84,
        'region': ee.Geometry.BBox(
            BOUNDS['west'], BOUNDS['south'],
            BOUNDS['east'], BOUNDS['north']
        ),
        'format': 'GEO_TIFF',
    })
    resp = requests.get(url, timeout=300)
    resp.raise_for_status()
    out = np.full(GRID_SHAPE, np.nan, dtype=np.float32)
    with MemoryFile(resp.content) as mem:
        with mem.open() as src:
            reproject(
                source=rasterio.band(src, 1), destination=out,
                src_transform=src.transform, src_crs=src.crs,
                dst_transform=GRID_TRANS, dst_crs=WGS84,
                resampling=Resampling.bilinear,
                src_nodata=src.nodata, dst_nodata=np.nan
            )
    return np.where(county_mask, out, np.nan)

print(f'Wards loaded: {len(wards)}')
print(f'WASI wards:   {len(wasi_wards)}')
print(f'Hotspot wards:{len(hotspot_wards)}')
print(f'Phase 2 coverage gap: {"available" if has_coverage else "not yet available"}')
print(f'Phase 2 boreholes:    {"available" if has_boreholes else "not yet available"}')


### 3. Figure 1 — Seasonal Water Availability Map

**Phase 1 deliverable:** Shows which water sources across Kitui are permanent year-round,
which are seasonal, and which are absent.

Data source: JRC Global Surface Water seasonality layer (1984 to 2021).
Each pixel shows the number of months per year that surface water was detected.

- 12 months = permanent water (rivers, reservoirs, large pans)
- 1 to 11 months = seasonal water (disappears during dry season)
- 0 months = no surface water detected

Communities depending on seasonal sources are most vulnerable during drought periods.


In [ ]:
# ── Figure 1: Seasonal water availability ─────────────────────────────────────
print('Loading JRC water seasonality from GEE Assets...')
water_seasonality = load_asset('kitui_jrc_water_seasonality', download_scale=30)
water_occurrence  = load_asset('kitui_jrc_water_occurrence',  download_scale=30)

ext = [BOUNDS['west'], BOUNDS['east'], BOUNDS['south'], BOUNDS['north']]

fig, axes = plt.subplots(1, 2, figsize=(20, 14))

# Panel A: seasonality
ax = axes[0]
# Custom colourmap: white=no water, light blue=seasonal, dark blue=permanent
cmap_water = mcolors.LinearSegmentedColormap.from_list(
    'water', ['#F5F5F5', '#9DC3E6', '#2E75B6', '#0B5394'], N=13
)
# Only show pixels with at least 1 month of water
seasonality_display = np.where(water_seasonality >= 1, water_seasonality, np.nan)
wards.plot(ax=ax, color='#F0EDE8', edgecolor='#AAAAAA', linewidth=0.4)
im = ax.imshow(seasonality_display, cmap=cmap_water, vmin=1, vmax=12,
               extent=ext, origin='upper', aspect='equal', zorder=2)
wards.boundary.plot(ax=ax, color='#888888', linewidth=0.4, zorder=3)
cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label('Months per year with surface water')
cbar.set_ticks([1, 3, 6, 9, 12])
cbar.set_ticklabels(['1 (very seasonal)', '3', '6', '9', '12 (permanent)'])
ax.set_title('Surface Water Seasonality — Kitui County\nJRC Global Surface Water | 1984 to 2021 | 30m resolution',
             fontsize=11, fontweight='bold')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
add_north_arrow(ax)
add_caption(ax, 'Blue = surface water detected. Darker = more months per year. White = no surface water.')

# Panel B: occurrence (reliability)
ax = axes[1]
occurrence_display = np.where(water_occurrence > 0, water_occurrence, np.nan)
wards.plot(ax=ax, color='#F0EDE8', edgecolor='#AAAAAA', linewidth=0.4)
im2 = ax.imshow(occurrence_display, cmap='Blues', vmin=0, vmax=100,
                extent=ext, origin='upper', aspect='equal', zorder=2)
wards.boundary.plot(ax=ax, color='#888888', linewidth=0.4, zorder=3)
cbar2 = plt.colorbar(im2, ax=ax, fraction=0.046, pad=0.04)
cbar2.set_label('% of observations with water present')
ax.set_title('Surface Water Occurrence (Reliability) — Kitui County\n% of Landsat observations 1984 to 2021 where water was detected',
             fontsize=11, fontweight='bold')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
add_north_arrow(ax)
add_caption(ax, 'High occurrence = reliable water source. Low occurrence = unreliable, appears only after rains.')

plt.suptitle('Seasonal Water Availability — Kitui County, Kenya',
             fontsize=14, fontweight='bold')
plt.tight_layout()
path = MAPS + 'fig01_seasonal_water_availability.png'
plt.savefig(path, dpi=300)
plt.show()
print(f'Figure 1 saved: {path}')


### 4. Figure 2 — Vegetation Stress and Land Condition Map

**Phase 1 deliverable:** Shows vegetation stress and land condition changes across Kitui
using 25 years of MODIS satellite data.

Two panels are produced:
- Current vegetation condition (2000 to 2025 mean NDVI)
- Vegetation anomaly (decline from the 2000 to 2004 baseline)

NDVI (Normalised Difference Vegetation Index) measures vegetation greenness from space.
Values closer to 1.0 indicate healthy dense vegetation.
Values closer to 0 indicate bare soil or severely stressed vegetation.

Areas showing significant decline below the early-period baseline represent locations
where land degradation and climate stress have reduced vegetation cover over time.


In [ ]:
# ── Figure 2: Vegetation stress and land condition ────────────────────────────
print('Loading NDVI layers from GEE Assets...')
ndvi_current  = load_asset('kitui_ndvi_mean_2000_2025',     download_scale=500)
ndvi_baseline = load_asset('kitui_ndvi_baseline_2000_2004', download_scale=500)

# NDVI anomaly: positive = decline below baseline (stress)
ndvi_anomaly = np.where(
    (~np.isnan(ndvi_current)) & (~np.isnan(ndvi_baseline)),
    ndvi_baseline - ndvi_current,
    np.nan
)

fig, axes = plt.subplots(1, 3, figsize=(24, 14))

ext = [BOUNDS['west'], BOUNDS['east'], BOUNDS['south'], BOUNDS['north']]

# Panel A: current NDVI
ax = axes[0]
valid = ndvi_current[~np.isnan(ndvi_current)]
im = ax.imshow(ndvi_current, cmap='RdYlGn', vmin=0.1, vmax=0.8,
               extent=ext, origin='upper', aspect='equal')
wards.boundary.plot(ax=ax, color='#333333', linewidth=0.4)
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label='NDVI (0=bare, 1=dense vegetation)')
ax.set_title('Current Vegetation Condition\nMean NDVI 2000 to 2025 (MODIS MOD13A3)',
             fontsize=10, fontweight='bold')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
add_north_arrow(ax)
add_caption(ax, f'Mean NDVI: {valid.mean():.3f} | Range: {valid.min():.3f} to {valid.max():.3f}')

# Panel B: baseline NDVI
ax = axes[1]
valid_b = ndvi_baseline[~np.isnan(ndvi_baseline)]
im2 = ax.imshow(ndvi_baseline, cmap='RdYlGn', vmin=0.1, vmax=0.8,
                extent=ext, origin='upper', aspect='equal')
wards.boundary.plot(ax=ax, color='#333333', linewidth=0.4)
plt.colorbar(im2, ax=ax, fraction=0.046, pad=0.04, label='NDVI (0=bare, 1=dense vegetation)')
ax.set_title('Baseline Vegetation Condition\nMean NDVI 2000 to 2004 (earliest MODIS window)',
             fontsize=10, fontweight='bold')
ax.set_xlabel('Longitude')
add_north_arrow(ax)
add_caption(ax, f'Mean NDVI: {valid_b.mean():.3f} | MODIS baseline starts 2000 — pre-2000 data unavailable')

# Panel C: anomaly (decline)
ax = axes[2]
valid_a = ndvi_anomaly[~np.isnan(ndvi_anomaly)]
vmax_a = np.percentile(np.abs(valid_a), 98)
im3 = ax.imshow(ndvi_anomaly, cmap='RdYlGn_r', vmin=-vmax_a, vmax=vmax_a,
                extent=ext, origin='upper', aspect='equal')
wards.boundary.plot(ax=ax, color='#333333', linewidth=0.4)
cbar3 = plt.colorbar(im3, ax=ax, fraction=0.046, pad=0.04)
cbar3.set_label('NDVI change (positive = decline below baseline)')
ax.set_title('Vegetation Stress — Decline from Baseline\nPositive values = vegetation worse than 2000-2004',
             fontsize=10, fontweight='bold')
ax.set_xlabel('Longitude')
add_north_arrow(ax)
pct_decline = (valid_a > 0).mean() * 100
add_caption(ax, f'{pct_decline:.1f}% of county shows vegetation below baseline | Red = degraded | Green = improved')

plt.suptitle('Vegetation Stress and Land Condition — Kitui County, Kenya\n25 Years of MODIS Satellite Data (2000-2025)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
path = MAPS + 'fig02_vegetation_stress.png'
plt.savefig(path, dpi=300)
plt.show()
print(f'Figure 2 saved: {path}')


### 5. Figure 3 — WASI Ward Choropleth

Produces the publication-ready ward-level WASI map for the report.
Labels the 10 highest-stress wards and shows the stress class distribution.


In [ ]:
# ── Figure 3: WASI ward choropleth ────────────────────────────────────────────

fig, ax = plt.subplots(figsize=(12, 14))

wasi_wards.plot(
    column='WASI_mean', cmap='RdYlGn_r', linewidth=0.6, edgecolor='white',
    legend=True, vmin=0, vmax=1,
    legend_kwds={'label': 'Water Access Stress Index (0 = low stress, 1 = high stress)',
                 'orientation': 'vertical', 'shrink': 0.7},
    ax=ax
)

# Label top 10 stress wards
top10 = wasi_wards.nlargest(10, 'WASI_mean')
for _, row in top10.iterrows():
    c = row.geometry.centroid
    ax.annotate(row['Ward'], xy=(c.x, c.y), fontsize=6.5,
                ha='center', va='center', fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.2', fc='white', alpha=0.6, ec='none'))

ax.set_title('Water Access Stress Index — Kitui County\n'
             'Phase 1 satellite analysis: 40 wards, 500m resolution',
             fontsize=12, fontweight='bold')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
add_north_arrow(ax)
add_caption(ax, 'C1 (distance to boreholes) not included — borehole dataset pending. '
                'Source: CHIRPS, MODIS, WorldPop, SRTM via Google Earth Engine.')

plt.tight_layout()
path = MAPS + 'fig03_wasi_choropleth.png'
plt.savefig(path, dpi=300)
plt.show()
print(f'Figure 3 saved: {path}')


### 6. Figure 4 — WASI Component Breakdown Chart

A horizontal stacked bar chart showing the contribution of each component to the WASI score for every ward.
Wards are ordered from lowest to highest total stress.
This chart is useful for understanding which stress driver is dominant in each ward.


In [ ]:
# ── Figure 4: WASI component breakdown chart ──────────────────────────────────

wasi_sorted = wasi_table.sort_values('WASI_mean', ascending=True).copy()

comp_cols   = ['C2_Rainfall', 'C3_NDVI', 'C4_Population', 'C5_Slope']
comp_labels = ['C2: Rainfall variability (35.7%)', 'C3: NDVI anomaly (21.4%)',
               'C4: Population (28.6%)', 'C5: Slope (14.3%)']
comp_weights = [0.357, 0.214, 0.286, 0.143]
comp_colors  = ['#2E75B6', '#70AD47', '#ED7D31', '#FFC000']

fig, ax = plt.subplots(figsize=(14, 12))
y_pos = np.arange(len(wasi_sorted))
cumulative = np.zeros(len(wasi_sorted))

for col, label, w, colour in zip(comp_cols, comp_labels, comp_weights, comp_colors):
    if col in wasi_sorted.columns:
        vals = wasi_sorted[col].fillna(0).values * w
        ax.barh(y_pos, vals, left=cumulative, height=0.7,
                color=colour, alpha=0.85, label=label)
        cumulative += vals

# WASI composite dot
ax.scatter(wasi_sorted['WASI_mean'].values, y_pos,
           color='black', s=20, zorder=5, label='WASI composite')

ax.set_yticks(y_pos)
ax.set_yticklabels(wasi_sorted['Ward'], fontsize=7.5)
ax.set_xlabel('Weighted stress contribution (0 = no stress, 1 = maximum stress)')
ax.set_title('WASI Component Breakdown by Ward — Kitui County\n'
             'Phase 1: four satellite components, C1 (boreholes) pending',
             fontsize=12, fontweight='bold')
ax.legend(loc='lower right', fontsize=8)
ax.axvline(0.55, color='red', linestyle='--', alpha=0.4, linewidth=1)
ax.text(0.555, len(wasi_sorted) - 1, 'High stress\nthreshold',
        fontsize=7, color='red', alpha=0.6)
ax.set_xlim(0, 0.75)
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
path = MAPS + 'fig04_wasi_components.png'
plt.savefig(path, dpi=300)
plt.show()
print(f'Figure 4 saved: {path}')


### 7. Figure 5 — Hotspot Analysis Map

Copies the hotspot map produced by Notebook 03 into the report figures folder
and regenerates a clean single-panel version for the report.


In [ ]:
# ── Figure 5: Hotspot analysis map ────────────────────────────────────────────
import shutil

# Copy the Notebook 03 output
src = OUT + 'kitui_hotspot_map.png'
if os.path.exists(src):
    shutil.copy(src, MAPS + 'fig05_hotspot_map.png')
    print(f'Figure 5 copied from Notebook 03 output')

# Regenerate clean single-panel ward classification map
WARD_COLOURS = {
    'Hotspot':         '#C00000',
    'Not significant': '#D9D9D9',
    'Coldspot':        '#2E75B6',
    'No data':         '#F0F0F0',
}

fig, ax = plt.subplots(figsize=(12, 14))

for cls, colour in WARD_COLOURS.items():
    subset = hotspot_wards[hotspot_wards['Ward_Class'] == cls] if 'Ward_Class' in hotspot_wards.columns else hotspot_wards
    if len(subset) > 0:
        subset.plot(ax=ax, color=colour, linewidth=0.5, edgecolor='white')

# Label hotspot wards
if 'Ward_Class' in hotspot_wards.columns:
    for _, row in hotspot_wards[hotspot_wards['Ward_Class'] == 'Hotspot'].iterrows():
        c = row.geometry.centroid
        ax.annotate(row['Ward'], xy=(c.x, c.y), fontsize=7,
                    ha='center', va='center', fontweight='bold', color='white',
                    bbox=dict(boxstyle='round,pad=0.15', fc='#C00000', alpha=0.7, ec='none'))

legend_patches = [
    mpatches.Patch(color='#C00000', label='Hotspot — statistically significant high-stress cluster'),
    mpatches.Patch(color='#D9D9D9', label='Not significant'),
    mpatches.Patch(color='#2E75B6', label='Coldspot — statistically significant low-stress cluster'),
]
ax.legend(handles=legend_patches, loc='lower left', fontsize=8)
ax.set_title('Water Stress Spatial Clustering — Kitui County\n'
             "Getis-Ord Gi* | 500m raster | Moran's I = 0.332 (p=0.001)",
             fontsize=12, fontweight='bold')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
add_north_arrow(ax)
add_caption(ax, 'Ward classified as hotspot if >=30% of pixels have Gi* z-score > 1.96. Source: WASI raster (Notebook 02).')

plt.tight_layout()
path = MAPS + 'fig05_hotspot_ward.png'
plt.savefig(path, dpi=300)
plt.show()
print(f'Figure 5 saved: {path}')


### 8. Phase 2 Figures (Pending Borehole Data)

The following figures will be added once the borehole dataset is received and Notebooks 04 and 05 have been run:

- **Figure 6** — Borehole network overview map (location, status, GPS quality)
- **Figure 7** — Coverage gap map (1km walking threshold, population in gap)
- **Figure 8** — Priority site rankings (top 20 pilot candidates)

This cell runs the Phase 2 figures if the outputs are available and skips them otherwise.


In [ ]:
# ── Phase 2 figures — runs if outputs available, skips otherwise ───────────────

if has_boreholes and has_coverage:
    print('Phase 2 outputs detected — generating Figures 6 and 7...')

    # Figure 6: Borehole network
    fig, ax = plt.subplots(figsize=(12, 14))
    wards.plot(ax=ax, color='#F5F5F5', edgecolor='#CCCCCC', linewidth=0.5)
    functional_bh = boreholes[boreholes['Functional'] == True] if 'Functional' in boreholes.columns else boreholes
    nonfunctional_bh = boreholes[boreholes['Functional'] == False] if 'Functional' in boreholes.columns else gpd.GeoDataFrame()
    functional_bh.plot(ax=ax, color='#2E75B6', markersize=5, alpha=0.7,
                       label=f'Functional ({len(functional_bh)})')
    if len(nonfunctional_bh) > 0:
        nonfunctional_bh.plot(ax=ax, color='#C00000', markersize=4, alpha=0.6,
                              marker='x', label=f'Non-functional ({len(nonfunctional_bh)})')
    ax.legend(fontsize=9, loc='lower left')
    ax.set_title('Borehole Network — Kitui County', fontsize=12, fontweight='bold')
    ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
    add_north_arrow(ax)
    plt.tight_layout()
    plt.savefig(MAPS + 'fig06_borehole_network.png', dpi=300)
    plt.show()
    print('Figure 6 saved')

    # Figure 7: Coverage gap choropleth
    fig, ax = plt.subplots(figsize=(12, 14))
    coverage_wards.plot(
        column='Pop_Gap_Pct_1km', cmap='Reds', linewidth=0.5, edgecolor='white',
        legend=True, vmin=0, vmax=100,
        legend_kwds={'label': '% population outside 1km borehole coverage', 'orientation': 'vertical'},
        ax=ax
    )
    ax.set_title('Population Coverage Gap — Kitui County\n1km walking threshold', fontsize=12, fontweight='bold')
    ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
    add_north_arrow(ax)
    plt.tight_layout()
    plt.savefig(MAPS + 'fig07_coverage_gap.png', dpi=300)
    plt.show()
    print('Figure 7 saved')

else:
    print('Phase 2 outputs not yet available.')
    print('Run Notebooks 04 and 05 after borehole data is received, then re-run this cell.')


### 9. Export Report Tables

Saves clean CSV versions of the key analysis tables for use in the report.
Column names are formatted for readability.


In [ ]:
# ── Export report tables ───────────────────────────────────────────────────────

# Table 1: WASI ward summary
t1 = wasi_table.sort_values('WASI_mean', ascending=False).copy()
t1.to_csv(REPT + 'table01_wasi_ward.csv', index=False)
print(f'Table 1 saved: table01_wasi_ward.csv ({len(t1)} wards)')

# Table 2: Coverage gap (Phase 2)
if has_coverage:
    coverage_table.to_csv(REPT + 'table02_coverage_gap.csv', index=False)
    print(f'Table 2 saved: table02_coverage_gap.csv')
else:
    print('Table 2 (coverage gap): not yet available — run Notebook 04')

# Summary
print()
print('Report figures complete')
print(f'Maps saved to: {MAPS}')
print()
phase1_figs = [f for f in os.listdir(MAPS) if f.startswith('fig0') and f.endswith('.png')]
for f in sorted(phase1_figs):
    print(f'  {f}')
print()
print('Phase 1 deliverables:')
print('  fig01_seasonal_water_availability.png  -- Seasonal water availability map')
print('  fig02_vegetation_stress.png            -- Vegetation stress and land condition map')
print('  fig03_wasi_choropleth.png              -- Water Access Stress Index map')
print('  fig04_wasi_components.png              -- WASI component breakdown')
print('  fig05_hotspot_ward.png                 -- Spatial hotspot analysis')
print()
print('Phase 2 figures will be added once borehole data is received.')
